[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%207/L13_Cobalt_Support_Copilot.ipynb)

# Cobalt Support Copilot — Part 1: the layer that **reads**
### ISBA 2411 · Week 7 · Lecture 13

---

### The situation

You just became the product manager for customer support at **Cobalt**.

**What Cobalt sells.** Software, on a monthly subscription, to about 600 other businesses. It plugs into
their company databases and turns the raw numbers into the charts and dashboards their staff check every
morning.

**Why customers write in.** When something breaks for one of those 600 companies (a chart shows a number
they know is wrong, nobody on their team can sign in, an invoice looks odd) they click **Help &rarr; Contact
support** inside the product, or email support@cobalt.com, and type out the problem in their own words.

**Where it lands.** Each of those messages becomes a **ticket**: the customer's text, plus a note of who
sent it, when, and which plan they are on. Every ticket drops into one shared work queue that three support
agents grind through, oldest first. **That queue is what we will call the inbox** for the rest of this
notebook. Not your email inbox: this one specific queue of customer problems waiting to be answered.

Two real ones, so you know what we are dealing with:

> *"Dashboards take 40+ seconds to load since yesterday. They used to come up instantly."*
> *"We were charged twice this month for the same seats. We need a refund on the duplicate."*

That is all a ticket is: a couple of sentences of ordinary writing. No form, no dropdown, no category.

**The scale.** ~5,000 arrive every week and three people answer them. Roughly **40%** are variations of the
same handful of questions, and an agent loses the first few minutes of every ticket just working out what it
is about and who should own it.

#### Why this is hard: there is no training data

The normal way to teach a computer to sort text is to hand it a few thousand examples **a person has already
sorted by hand** &mdash; somebody reads ticket 1 and writes `billing`, reads ticket 2 and writes `login`, a few
thousand times over. That pile of hand-sorted examples is called **training data**, and most machine learning
cannot start without it.

Cobalt has none. Building it would mean paying people to read and tag thousands of tickets *before* you could
write a line of code. That is weeks of work and a budget nobody approved.

**So the textbook answer is off the table.** Everything below is about building something useful anyway.

**Today we build the part of the product that reads the inbox** — search it, sort it, and decide
what can be handled automatically versus what needs a human. Next lecture we make it *act*.

---

### How to use this notebook

You will **not** write code from scratch. Every cell is already written — you just run it.

- 🔮 **Before you run this, predict:** — stop and commit to an answer. This is the part that makes it stick.
- 🎛️ **YOUR KNOB** — one thing you change, then re-run, and watch what happens.
- ✅ **What just happened** — the plain-language explanation.
- 💼 **At work this means** — why anyone would pay you for this.

### ⚠️ First: switch on the GPU

**Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save.**

Without it the model cells take several minutes instead of several seconds.

In [ ]:
# ==================  CELL 1  ==================
%pip install -q transformers sentence-transformers sentencepiece scikit-learn pandas matplotlib

import pandas as pd, numpy as np, matplotlib.pyplot as plt, torch, textwrap
pd.set_option("display.max_colwidth", 90)

DEVICE = 0 if torch.cuda.is_available() else -1
print("GPU available:", torch.cuda.is_available(), "\n")

tickets = pd.read_csv("https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/data/cobalt_tickets.csv")
print(f"{len(tickets):,} tickets loaded")
tickets[["ticket_id","created_at","plan","priority","ticket_text"]].head()

---
## Act 0 · The inbox you just inherited

Before any technology, look at the problem. This is the whole job: **5,000 of these a week, and nobody
has told you what any of them are about.**

Notice there is **no category column** in what we loaded. That is the honest starting position for
almost every real text project — you have the text, and nothing else.

In [ ]:
# ==================  CELL 2  ==================
# What does the raw inbox actually look like?
for t in tickets.ticket_text.sample(6, random_state=7):
    print(" •", textwrap.shorten(t, 95))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.2))
tickets.priority.value_counts().reindex(["urgent","high","normal","low"]).plot(
    kind="bar", ax=ax[0], color="#4F46E5"); ax[0].set_title("Tickets by priority")
tickets.plan.value_counts().plot(kind="bar", ax=ax[1], color="#059669"); ax[1].set_title("By customer plan")
tickets.ticket_text.str.split().str.len().plot(kind="hist", bins=14, ax=ax[2], color="#B45309")
ax[2].set_title("Ticket length (words)")
for a in ax: a.set_xlabel(""); a.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

✅ **What just happened.** Short, messy, human text — a median of about 13 words, no structure,
nothing sorted. A keyword rule would drown in this.

💼 **At work this means:** the first question on any text project is not "which model?" — it's
**"what do I actually have?"** Here: text, and nothing else. That single fact rules out most
traditional analytics and points straight at the technology we're about to use.

---
## Act 1 · Teach the computer to *read*

A computer cannot compare two sentences the way you can. Our first job is turning each ticket into
something comparable — an **embedding**: a list of numbers that stands for the ticket's *meaning*.

Once every ticket is a point in the same space, "find me similar tickets" becomes arithmetic.

🔮 **Before you run this, predict:** We're going to search for **"we are being charged wrong"**.
Not one ticket in this inbox contains that exact phrase. Will the search find the billing complaints anyway?

In [ ]:
# ==================  CELL 3  ==================
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")
E = encoder.encode(tickets.ticket_text.tolist(), normalize_embeddings=True)
print("every ticket is now a vector:", E.shape, "  (tickets x numbers)")

def keyword_search(q, k=3):
    hits = tickets[tickets.ticket_text.str.contains(q, case=False, na=False)]
    return hits.head(k)

def semantic_search(q, k=3):
    sims = E @ encoder.encode([q], normalize_embeddings=True)[0]
    top = np.argsort(-sims)[:k]
    return tickets.iloc[top].assign(similarity=sims[top].round(2))

QUERY = "we are being charged wrong"
print(f"\nKEYWORD search for {QUERY!r} -> {len(keyword_search(QUERY))} results")
print(f"SEMANTIC search for {QUERY!r}:")
for _, r in semantic_search(QUERY).iterrows():
    print(f"   {r.similarity:.2f}  {textwrap.shorten(r.ticket_text, 88)}")

✅ **What just happened.** Keyword search found **nothing** — no ticket uses that phrasing. Semantic
search returned the billing complaints anyway, including *"We were charged twice this month for the
same seats."* The word "charged" matched, but so did *invoice*, *refund*, *trial* — because the model
places them near each other **by meaning**, not spelling.

💼 **At work this means:** every "search that actually works" feature you've used — help centres,
support tools, internal wikis — is this. It's also the foundation of the retrieval systems we build
in Week 8.

### 🎛️ YOUR KNOB — search the inbox yourself

Change `MY_QUERY` below to anything a support manager might ask, then re-run the cell. Try:
`"customers who want to cancel"` · `"something is broken after your update"` · `"security worries"`

In [ ]:
# ==================  CELL 4  ==================
MY_QUERY = "customers who want to cancel"     # <-- change me, then re-run

for _, r in semantic_search(MY_QUERY, k=5).iterrows():
    print(f"{r.similarity:.2f}  [{r.priority:6}] {textwrap.shorten(r.ticket_text, 85)}")

### The shape of the whole inbox

If every ticket is a point, we can flatten those points onto a page and *look* at the inbox.

In [ ]:
# ==================  CELL 5  ==================
from sklearn.decomposition import PCA
xy = PCA(n_components=2, random_state=0).fit_transform(E)

plt.figure(figsize=(8.5, 6))
plt.scatter(xy[:,0], xy[:,1], s=34, c="#94A3B8", alpha=.75)
for q, col in [("billing and invoices","#E11D48"), ("cannot sign in","#4F46E5"),
               ("the app is slow","#059669"), ("how do I do this","#B45309")]:
    s = E @ encoder.encode([q], normalize_embeddings=True)[0]
    top = np.argsort(-s)[:14]
    plt.scatter(xy[top,0], xy[top,1], s=52, c=col, label=q)
plt.legend(fontsize=9); plt.xticks([]); plt.yticks([])
plt.title("The inbox, arranged by meaning — neighbourhoods form on their own")
plt.tight_layout(); plt.show()

✅ **What just happened.** Nobody sorted anything, yet tickets about the same thing landed near each
other. **Structure was already in the text** — we just made it visible.

💼 **At work this means:** you can show a leadership team what's in a pile of feedback *before*
spending a dollar on hand-sorting.

---
### 🔬 Under the hood — what *is* an embedding?

*Optional deep dive. Skip it and the product story still works — but this is where "magic" turns into
four lines of arithmetic you could do yourself.*

`encoder.encode()` did three things. Let's do each one by hand.

In [ ]:
# ==================  CELL 6  ==================
st_tok = encoder.tokenizer                 # the tokenizer inside the encoder
bert   = encoder[0].auto_model             # the transformer inside the encoder

sample = "Our Postgres connector hasn't pulled new rows since Tuesday."

# STEP 1 - the model never sees words. It sees TOKENS, then integer ids.
print("text  :", sample)
print("tokens:", st_tok.tokenize(sample))
print("ids   :", st_tok(sample)["input_ids"][:12], "...")

Notice the tokens are word-*pieces*, not words. A fixed vocabulary (~30k pieces) can spell anything by
gluing pieces together — which is how the model handles a word it has never seen.

**Does your business vocabulary survive?** Worth checking on any real project:

In [ ]:
# ==================  CELL 7  ==================
for w in ["Okta", "SAML", "webhook", "Snowflake", "MFA", "BigQuery", "Kubernetes"]:
    pieces = st_tok.tokenize(w)
    flag = "one token" if len(pieces) == 1 else f"SPLIT into {len(pieces)}"
    print(f"  {w:12} -> {str(pieces):42} {flag}")

✅ **Look at what happened to your product vocabulary.** Almost none of it survives: `Okta` becomes
`ok` + `##ta`, `webhook` becomes `web` + `##ho` + `##ok`, `Kubernetes` shatters into four fragments.

This model's vocabulary was built from general web text, and it has **never seen your industry's
words**. It is spelling them out phonetically, like someone sounding out a foreign name. `Snowflake`
isn't a data warehouse to this model — it's `snow` + `##fl` + `##ake`.

💼 **At work this means:** semantic search degrades exactly where your business is most specific —
product names, error codes, drug names, part numbers, legal citations. Run this two-minute check
before promising anyone that "AI search" will work on specialist text. When it matters, the fixes are
to fine-tune on your domain (next lecture) or keep a keyword search alongside the semantic one.

#### From tokens to ONE vector per ticket

The transformer gives back a vector *per token*. But we need one vector per **ticket**. The trick is
almost disappointingly simple: **average them**.

In [ ]:
# ==================  CELL 8  ==================
import torch

device_t = next(bert.parameters()).device
batch = st_tok(sample, return_tensors="pt", truncation=True)
batch = {k: v.to(device_t) for k, v in batch.items()}

with torch.no_grad():
    token_vectors = bert(**batch).last_hidden_state      # (1, n_tokens, 384)
print("one vector PER TOKEN :", tuple(token_vectors.shape))

# average across tokens (ignoring padding), then scale to length 1
mask   = batch["attention_mask"].unsqueeze(-1)
pooled = (token_vectors * mask).sum(1) / mask.sum(1)
pooled = torch.nn.functional.normalize(pooled, dim=1)
print("after mean-pooling   :", tuple(pooled.shape), " <- one vector for the whole ticket")

library = encoder.encode([sample], normalize_embeddings=True)
print("\nsame as the library? max difference =",
      float(np.abs(pooled.cpu().numpy() - library).max()))

✅ **What just happened.** The one-line `encoder.encode()` is: tokenize → run the transformer → average
the token vectors → normalize. Our by-hand version reproduces the library's answer **exactly**.

⚠️ And notice the weakness this exposes: a ticket's vector is an **average**. A long ticket covering
three different problems gets blended into one mushy point — which is exactly why very long documents
are usually *chunked* before being embedded (a technique we'll need in Week 8).

---
### 🔬 Under the hood — what happened inside `bert(...)`?

*You just called a transformer and got vectors back. Two things happen in there, over and over.*

**Attention: look around.** Every word reads every other word in the ticket and pulls in what it needs.
Information moves **between** words.

**Feed-forward: think about it.** Each word then goes through a small network **on its own**, which
applies what the model knows about it. No looking sideways.

That pair is one **block**, and this model stacks 6 of them (the big ones stack 12 to 96). That is all
"a deep model" means: the same two steps, run again and again.

🔮 **Before you run this, predict:** the word "plan" means a *subscription tier* in one ticket and an
*intention* in another. Going in, both are the same word, so both start as the same vector. Will they
still be the same coming out?

In [ ]:
# ==================  CELL 9  ==================
A = "We want to move to the Team plan next month."     # plan = a subscription tier
B = "We plan to migrate our database next week."       # plan = an intention

def vector_for(sentence, word, after_layers):
    b = st_tok(sentence, return_tensors="pt")
    b = {k: v.to(device_t) for k, v in b.items()}
    pos = st_tok.convert_ids_to_tokens(st_tok(sentence)["input_ids"]).index(word)
    with torch.no_grad():
        out = bert(**b, output_hidden_states=True)
    return out.hidden_states[after_layers][0, pos]

def alike(a, b):
    return float(a @ b / (a.norm() * b.norm()))

n_layers = bert.config.num_hidden_layers
print(f'how alike are the two "plan" vectors?\n')
sims = []
for L in range(n_layers + 1):
    s = alike(vector_for(A, "plan", L), vector_for(B, "plan", L))
    sims.append(s)
    tag = "  <- going in (same word, same vector)" if L == 0 else ("  <- coming out" if L == n_layers else "")
    print(f"  after {L} rounds of attention + feed-forward:  {s:.2f}  {'#' * int(s*38)}{tag}")

✅ **What just happened.** Going in, the two vectors are nearly identical, because at that point it is
literally the same word looking up the same row in the same table. The model has no idea there is a
difference.

Then each round of attention lets "plan" read its neighbours. *Team* and *month* pull the first one one
way; *migrate* and *database* pull the other one the other way. By the end they are barely related.

**That gap is attention doing its job.** And it is why the search back in Act 1 worked: it was never
matching spellings, it was matching vectors that already knew what the sentence was about.

#### The other half: where the knowledge is kept

Attention mixes words together. The **feed-forward** half applies what the model has stored. Let us
check that it has stored anything: blank out a word and ask it to fill the gap.

In [ ]:
# ==================  CELL 10  ==================
from transformers import pipeline
fill = pipeline("fill-mask", model="bert-base-uncased")

for guess in fill("Our subscription is billed every [MASK].")[:3]:
    print(f"  {guess['token_str']:8} {guess['score']:.2f}")

# and where do the model's weights actually sit?
att = ffn = 0
for name, p in bert.named_parameters():
    if "layer" not in name: continue
    if ".attention." in name: att += p.numel()
    elif ".intermediate." in name or ".output." in name: ffn += p.numel()
print(f"\n  attention weights     {att/1e6:5.1f}M   ({att/(att+ffn):.0%})")
print(f"  feed-forward weights  {ffn/1e6:5.1f}M   ({ffn/(att+ffn):.0%})")

✅ **What just happened.** Nobody trained this on Cobalt, and nobody told it what a subscription is,
but it knows they are billed **monthly**. That knowledge is not in the attention half; it is sitting in
the feed-forward weights.

And look at the split: about **two thirds** of every layer is feed-forward, only a third is attention.
Attention gets all the headlines. The majority of the model, by weight, is the part that stores what it
knows.

💼 **At work this means:** when someone says a model "understands" your text, this is the machinery
they mean. Attention supplies the context, the feed-forward half supplies the world knowledge, and
neither one was trained on your company.

#### And "similarity" is just multiplication

Because every vector is scaled to length 1, similarity is a plain **dot product** — multiply
matching numbers, add them up. That's the entire search engine.

In [ ]:
# ==================  CELL 11  ==================
a = encoder.encode(["I can't log in to my account"],        normalize_embeddings=True)[0]
b = encoder.encode(["Password reset email never arrives"],   normalize_embeddings=True)[0]
c = encoder.encode(["Please add dark mode to the mobile app"], normalize_embeddings=True)[0]

print("by hand  (a . b):", round(float(sum(a * b)), 3), " <- both about sign-in")
print("by hand  (a . c):", round(float(sum(a * c)), 3), " <- unrelated topics")
print("\nsame numbers, computed with numpy:", np.round([a @ b, a @ c], 3))

#### One constraint that bites in production

In [ ]:
# ==================  CELL 12  ==================
print("layers          :", bert.config.num_hidden_layers)
print("vector size     :", bert.config.hidden_size)
print("vocabulary      :", f"{bert.config.vocab_size:,} pieces")
print("MAX INPUT LENGTH:", st_tok.model_max_length, "tokens\n")

long_ticket = ("Following up on my earlier message. " * 40) + "THE ACTUAL PROBLEM IS BILLING."
n_tokens = len(st_tok(long_ticket)["input_ids"])
kept     = st_tok.decode(st_tok(long_ticket, truncation=True)["input_ids"][-14:])
print(f"a {n_tokens}-token ticket gets truncated; the tail the model actually keeps ends with:")
print("   ...", kept)
print("\nIs 'THE ACTUAL PROBLEM IS BILLING' still there?",
      "BILLING" in st_tok.decode(st_tok(long_ticket, truncation=True)["input_ids"]))

✅ **What just happened.** Every model has a hard input limit. Text past it is **silently discarded** —
no error, no warning. Here the customer's actual point was at the end of a rambling ticket, and it got
cut off.

💼 **At work this means:** when someone says "the AI ignored half my document," this is usually why.
Know your model's limit, and chunk long inputs rather than hoping.

---
## Act 2 · Route the inbox — with **no training data at all**

Now the product question: **can we sort tickets into the buckets our support team actually uses?**

The traditional answer is "hand-sort a few thousand tickets first, then train a classifier." That's weeks
of work and money you don't have.

Instead we'll use **zero-shot classification**: we describe each category *in plain English* and the
model decides which description fits. No training. Nobody sorts a ticket. It works because the model was already
pretrained on enormous amounts of text — someone else paid for that, and we get it for free.

🔮 **Before you run this, predict:** with **zero** training examples, what accuracy would you bet on?
Random guessing across 8 buckets would be about 12%.

In [ ]:
# ==================  CELL 13  ==================
from transformers import pipeline

# Describe each bucket in plain English. THIS IS THE 'PROGRAM'.
CATEGORIES = {
 "login_access"   : "signing in, passwords, two-factor authentication or account access",
 "billing_plan"   : "an invoice, payment, refund, subscription plan or pricing question",
 "data_sync"      : "a database connector failing to refresh or import data",
 "integrations"   : "connecting to another product such as Slack, Salesforce, webhooks or the API",
 "performance"    : "the product being slow, timing out or hanging",
 "bug_ui"         : "a chart, table or screen displaying something incorrectly",
 "how_to"         : "asking for instructions on how to do something",
 "feature_request": "requesting a new capability that does not exist yet",
}

ZS_MODEL = "MoritzLaurer/deberta-v3-base-zeroshot-v2.0"
router = pipeline("zero-shot-classification", model=ZS_MODEL, device=DEVICE)

descriptions = list(CATEGORIES.values())
back = {d: k for k, d in CATEGORIES.items()}

out = router(tickets.ticket_text.tolist(), candidate_labels=descriptions, multi_label=False)
routed = tickets.assign(
    predicted =[back[o["labels"][0]] for o in out],
    confidence=[round(o["scores"][0], 2) for o in out],
    second    =[back[o["labels"][1]] for o in out],
)
print("Routed", len(routed), "tickets. Tickets hand-sorted to make this work:", 0, "\n")
routed[["ticket_text","predicted","confidence"]].head(8)

In [ ]:
# ==================  CELL 14  ==================
fig, ax = plt.subplots(1, 2, figsize=(14, 3.6))
routed.predicted.value_counts().plot(kind="barh", ax=ax[0], color="#4F46E5")
ax[0].set_title("Where the copilot sent the tickets"); ax[0].invert_yaxis()
ax[1].hist(routed.confidence, bins=18, color="#059669")
ax[1].set_title("How sure was it?"); ax[1].set_xlabel("confidence")
plt.tight_layout(); plt.show()

### Now let's grade it

The dataset *does* have a hidden answer key (someone sorted these when the data was made, purely so we
could grade). We held it back so the routing above was honest.
Time to see how a system with **no training data** actually did.

In [ ]:
# ==================  CELL 15  ==================
from sklearn.metrics import accuracy_score

top1 = accuracy_score(tickets.category, routed.predicted)
top2 = np.mean([t in (p, s) for t, p, s in zip(tickets.category, routed.predicted, routed.second)])

print(f"  random guessing        {1/len(CATEGORIES):.1%}")
print(f"  top-1 accuracy         {top1:.1%}   <- the copilot's single best guess")
print(f"  top-2 accuracy         {top2:.1%}   <- correct answer in its top two")
print(f"\n  tickets hand-sorted: 0")
print(f"  models trained:         0")

> ### ⚠️ A confusing name
> The library calls that argument **`candidate_labels`**. Those are **not** the hand-sorted labels we said
> we do not have. Nobody sorted a single ticket. They are the sentences *you wrote*, in English, thirty
> seconds ago. The library's naming is unfortunate; the distinction matters.
>
> | | |
> |---|---|
> | **Training data** (what Cobalt lacks) | thousands of tickets a person read and tagged by hand |
> | **Category descriptions** (what we wrote) | eight sentences, written once, in about a minute |

✅ **What just happened.** From **nothing** — no training data, no training run, no GPU-hours — the copilot sorts
the inbox far better than chance, and its top-two suggestions contain the right answer the large
majority of the time.

Read that second number as a product, not a metric: *"show the agent two buttons instead of eight."*

💼 **At work this means:** you can ship a useful v1 **this week**, and only then decide whether paying
people to hand-sort tickets is worth it. The expensive step is now the *last* resort instead of the first.

### 🎛️ YOUR KNOB — the category descriptions *are* the program

There is no training here, so the only thing you control is **how you describe each bucket**. That
makes wording a design decision, not a detail.

Below, the descriptions have been replaced with lazy one-word versions. Run it and compare the accuracy
to what you just got. Then try writing better ones yourself.

In [ ]:
# ==================  CELL 16  ==================
LAZY = {"login_access":"login", "billing_plan":"billing", "data_sync":"sync",
        "integrations":"integrations", "performance":"performance", "bug_ui":"bug",
        "how_to":"how to", "feature_request":"feature request"}   # <-- rewrite these and re-run

d2 = list(LAZY.values()); back2 = {d: k for k, d in LAZY.items()}
out2 = router(tickets.ticket_text.tolist(), candidate_labels=d2, multi_label=False)
lazy_pred = [back2[o["labels"][0]] for o in out2]

print(f"  careful descriptions       : {top1:.1%}")
print(f"  lazy one-word descriptions : {accuracy_score(tickets.category, lazy_pred):.1%}")
print("\n  Same model. Same tickets. Only the wording changed.")

---
### 🔬 Under the hood — zero-shot isn't classification at all

*You just watched wording change accuracy by ~17 points. That should feel strange: in a normal
classifier the category is just a number, and what you call it makes no difference at all. Here the words
did real work. Why?*

**Because this model is not a classifier. It's a fact-checker.**

It was trained on a task called **Natural Language Inference (NLI)**: given a *premise* and a
*hypothesis*, decide whether the premise makes the hypothesis true (**entailment**) or not.

The zero-shot trick is to smuggle classification into that format:

> **premise** = the ticket
> **hypothesis** = `"This example is {your category description}."`

It scores every category's sentence and picks whichever one it most believes. Let's watch it happen.

In [ ]:
# ==================  CELL 17  ==================
from transformers import AutoTokenizer, AutoModelForSequenceClassification

nli_tok = AutoTokenizer.from_pretrained(ZS_MODEL)
nli     = AutoModelForSequenceClassification.from_pretrained(ZS_MODEL).eval()
print("what this model was actually trained to output:", nli.config.id2label, "\n")

ticket = "We were charged twice this month for the same seats. We need a refund on the duplicate."
print("PREMISE (the ticket):", ticket, "\n")

for cat in ["billing_plan", "performance"]:
    hypothesis = f"This example is {CATEGORIES[cat]}."
    pair = nli_tok(ticket, hypothesis, return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = nli(**pair).logits[0]
    verdict = nli.config.id2label[int(logits.argmax())]
    print(f"HYPOTHESIS: 'This example is {CATEGORIES[cat][:46]}...'")
    print(f"   logits  : entailment {logits[0]:+.2f} | not_entailment {logits[1]:+.2f}")
    print(f"   verdict : {verdict.upper()}\n")

✅ **What just happened.** The model read the ticket and judged **two sentences we wrote** as true or
false. For the billing sentence the entailment score is strongly positive; for the performance sentence
the sign flips. Turn those scores into percentages across all eight sentences and you get the
"confidence" from earlier.

**Now the description-wording knob makes sense.** You weren't naming a category — you were
**writing the sentence the model has to fact-check**. A vague sentence like *"billing"* is hard to
judge true or false. *"An invoice, payment, refund, subscription plan or pricing question"* is easy.

💼 **At work this means:** this is the same skill as writing a good prompt. Being precise about what
you're asking is not a soft skill here — it is literally the program. And it explains the whole family
of "just describe what you want" AI tools you're seeing: underneath, a model is scoring your words.

*(Footnote for the curious: the wrapper sentence itself — `"This example is {}."` — barely matters.
We tested three variants and accuracy moved less than a point. It's **your description** that carries
the weight, not the template around it.)*

---
## Act 3 · Where v1 breaks — and what a PM does about it

An average is a bad way to run a product. **Which** tickets does it get wrong, and does that matter?

In [ ]:
# ==================  CELL 18  ==================
recall = (routed.assign(ok=routed.predicted.values == tickets.category.values)
          .groupby(tickets.category).ok.mean().sort_values())

plt.figure(figsize=(8, 3.4))
colors = ["#E11D48" if v < .5 else "#059669" for v in recall.values]
plt.barh(recall.index, recall.values, color=colors)
plt.axvline(.5, ls="--", c="#334155", lw=1)
plt.title("How often each bucket is caught correctly"); plt.xlim(0, 1)
plt.tight_layout(); plt.show()

worst = recall.index[0]
print(f"Worst bucket: {worst}\n")
for _, r in routed[tickets.category.values == worst].head(4).iterrows():
    print(f"  sent to '{r.predicted}' ({r.confidence}) <- {textwrap.shorten(r.ticket_text, 78)}")

✅ **What just happened.** The failures are not random. The weak buckets are the ones whose
**descriptions overlap** — a failing data sync genuinely *is* "something is broken," and an API problem
genuinely *is* "an integration." The model isn't confused; **our categories are.**

💼 **At work this means:** when a classifier underperforms, the first thing to examine is usually your
**taxonomy**, not your model. Categories should describe *what the customer wants done*, not which
internal component is involved.

---
### 🔬 Under the hood — why was it **99% sure** and wrong?

*Look again at those misroutes. The copilot didn't hedge — it was 0.94, 0.99 confident, and wrong.
That is far more dangerous than being unsure, and the reason is worth understanding.*

🔮 **Before you run this, predict:** what happens if we ask the model to route a ticket about
something that isn't in our list at all — say, a job application?

In [ ]:
# ==================  CELL 19  ==================
off_topic = "Hi, I saw your careers page and I'd like to apply for the sales engineer role."
result = router(off_topic, candidate_labels=list(CATEGORIES.values()), multi_label=False)

print("TICKET:", off_topic, "\n")
for lab, sc in list(zip(result["labels"], result["scores"]))[:4]:
    print(f"   {sc:5.1%}  {lab[:62]}")
print(f"\n   scores sum to: {sum(result['scores']):.2f}")

✅ **What just happened.** A ticket that belongs in **none** of our eight buckets still got assigned to
one — with real confidence. And look at the last line: **the scores are forced to add up to 1.**

That's the mechanism. The final step (`softmax`) takes whatever scores exist and spreads 100% of the
belief across **the options you supplied**. There is no "none of these" outcome unless you build one.
So the number isn't *"how likely am I to be right"* — it's *"of the choices you gave me, which fits
best."* A confident wrong answer is what you get when every option is bad.

💼 **At work this means:** never show a raw confidence score to a user as if it were a probability of
correctness, and never let one gate an irreversible action. Two practical defences:

In [ ]:
# ==================  CELL 20  ==================
# Defence 1: give the model an escape hatch.
with_escape = dict(CATEGORIES)
with_escape["other"] = "something unrelated to using the product, such as sales, careers or spam"

r2 = router(off_topic, candidate_labels=list(with_escape.values()), multi_label=False)
back2 = {v: k for k, v in with_escape.items()}
print("WITH an 'other' bucket ->", back2[r2["labels"][0]], f"({r2['scores'][0]:.0%})\n")

# Defence 2: don't ask it to choose. Score each category independently (multi_label=True),
# so 'low on everything' is now a possible answer.
r3 = router(off_topic, candidate_labels=list(CATEGORIES.values()), multi_label=True)
print("Scored INDEPENDENTLY (multi_label=True) - top 3:")
for lab, sc in list(zip(r3["labels"], r3["scores"]))[:3]:
    print(f"   {sc:5.1%}  {lab[:58]}")
print(f"\n   these do NOT sum to 1 ({sum(r3['scores']):.2f}) - low everywhere = 'I don't know'")

✅ **What just happened.** Two one-line fixes, both worth knowing. Adding an **"other"** category gives
the model somewhere honest to put the odd ticket. Switching to `multi_label=True` scores each category
on its own merits instead of forcing a competition — so "low on everything" becomes a possible, and
very informative, answer.

💼 **At work this means:** the difference between a demo and a product is often just this — designing
what happens when the model is **out of its depth**. Ask any vendor: *"what does your system do with an
input it has never seen?"*

### 🎛️ YOUR KNOB — the decision that actually ships the product

You do not have to route everything. Route only what the copilot is **sure** about, and send the rest
to a human. That one threshold decides how much work you save and how many mistakes you make.

Move `THRESHOLD` and watch the trade-off.

In [ ]:
# ==================  CELL 21  ==================
THRESHOLD = 0.90        # <-- change me (try 0.5, 0.7, 0.95) and re-run

auto = routed.confidence >= THRESHOLD
acc_auto = accuracy_score(tickets.category[auto], routed.predicted[auto]) if auto.sum() else 0

print(f"  Auto-routed          : {auto.mean():.0%} of the inbox")
print(f"  Accuracy on those    : {acc_auto:.0%}")
print(f"  Sent to a human      : {(~auto).mean():.0%}")
print(f"  Tickets/week handled : ~{int(5000*auto.mean()):,} of 5,000")

grid = np.arange(.3, .99, .02)
cov = [(routed.confidence >= t).mean() for t in grid]
acc = [accuracy_score(tickets.category[routed.confidence >= t],
                      routed.predicted[routed.confidence >= t])
       if (routed.confidence >= t).sum() > 5 else np.nan for t in grid]
plt.figure(figsize=(7.5, 4))
plt.plot(grid, cov, label="share of inbox auto-routed", lw=2.5, color="#4F46E5")
plt.plot(grid, acc, label="accuracy of those decisions", lw=2.5, color="#059669")
plt.axvline(THRESHOLD, ls="--", c="#E11D48", label=f"your threshold ({THRESHOLD})")
plt.xlabel("confidence threshold"); plt.ylim(0, 1); plt.legend(); plt.grid(alpha=.25)
plt.title("The product decision: cover more, or be more right")
plt.tight_layout(); plt.show()

✅ **What just happened.** The two lines move in opposite directions, and **there is no setting that
maximises both.** Where you stand on that curve is a business judgement: how costly is a misrouted
ticket, versus how valuable is an agent's hour?

💼 **At work this means:** this curve — not accuracy — is what you take into the room. "We can
automate 44% of the inbox at 81% accuracy, or 87% of it at 66%" is a decision an executive can
actually make.

---
## What we shipped today

| | |
|---|---|
| **Semantic search** over the whole inbox | finds tickets by *meaning*, not keywords |
| **A router** into 8 buckets | built with **zero** training data |
| **A confidence policy** | auto-handle the easy ones, escalate the rest |
| **Cost of hand-sorting tickets** | **$0** |

### The one idea to take with you

> Someone else already spent millions of dollars teaching a model to read English.
> **Your job is not to build intelligence — it's to point it at your problem, cheaply, and know where it breaks.**

### Next lecture

Our router is useful but blunt: it needs a human for most of the inbox, and it can't pull out *what
specifically* is broken. Next time we ask what changes when you **do** invest in hand-sorting a few
hundred tickets, how the pretraining that makes all this possible actually worked — and where this
technology confidently makes things up.